In [ ]:
🧩 Problem Setup (Common for all)

We represent the board as a tuple (important for hashing in sets/dicts).

initial = (1,2,3,
           4,0,6,
           7,5,8)

goal = (1,2,3,
        4,5,6,
        7,8,0)

👉 0 = blank

🔁 Helper Functions (VERY IMPORTANT)
1️⃣ Get neighbors (valid moves)
def get_neighbors(state):
    neighbors = []
    index = state.index(0)  # position of blank
    x, y = index // 3, index % 3

    moves = [(-1,0),(1,0),(0,-1),(0,1)]

    for dx, dy in moves:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            new_index = nx*3 + ny
            new_state = list(state)

            # swap
            new_state[index], new_state[new_index] = new_state[new_index], new_state[index]

            neighbors.append(tuple(new_state))

    return neighbors
🟦 1. BFS (Shortest path guaranteed)
from collections import deque

def bfs_8puzzle(start, goal):
    queue = deque([(start, [])])
    visited = set()

    while queue:
        state, path = queue.popleft()

        if state == goal:
            return path

        visited.add(state)

        for neighbor in get_neighbors(state):
            if neighbor not in visited:
                queue.append((neighbor, path + [neighbor]))

    return None

✅ Guarantees minimum moves
❌ Memory heavy

🟨 2. DFS (Not recommended but important)
def dfs_8puzzle(start, goal):
    stack = [(start, [])]
    visited = set()

    while stack:
        state, path = stack.pop()

        if state == goal:
            return path

        if state not in visited:
            visited.add(state)

            for neighbor in get_neighbors(state):
                stack.append((neighbor, path + [neighbor]))

    return None

✅ Less memory
❌ Not optimal
❌ Can go deep unnecessarily

🟩 3. A* Search (BEST APPROACH)
Heuristic: Manhattan Distance
def manhattan(state, goal):
    distance = 0
    for i in range(9):
        if state[i] != 0:
            goal_index = goal.index(state[i])
            x1, y1 = i // 3, i % 3
            x2, y2 = goal_index // 3, goal_index % 3
            distance += abs(x1 - x2) + abs(y1 - y2)
    return distance
A* Implementation
import heapq

def a_star_8puzzle(start, goal):
    pq = [(0, start, [])]
    g_cost = {start: 0}

    while pq:
        _, state, path = heapq.heappop(pq)

        if state == goal:
            return path

        for neighbor in get_neighbors(state):
            temp_g = g_cost[state] + 1

            if neighbor not in g_cost or temp_g < g_cost[neighbor]:
                g_cost[neighbor] = temp_g
                f_cost = temp_g + manhattan(neighbor, goal)
                heapq.heappush(pq, (f_cost, neighbor, path + [neighbor]))

    return None

✅ Fast
✅ Optimal
✅ Most used in real applications

🔥 Run Example
solution = a_star_8puzzle(initial, goal)

for step in solution:
    print(step)



🟩 RBFS Code
def rbfs(start, goal):

    def recursive_rbfs(state, g, f_limit, path, visited):
        if state == goal:
            return path, g

        neighbors = get_neighbors(state)
        successors = []

        for neighbor in neighbors:
            if neighbor not in visited:
                new_g = g + 1
                h = manhattan(neighbor, goal)
                f = new_g + h
                successors.append((f, neighbor, new_g))

        if not successors:
            return None, float('inf')

        while True:
            # sort by f value
            successors.sort(key=lambda x: x[0])

            best_f, best_state, best_g = successors[0]

            if best_f > f_limit:
                return None, best_f

            # second best f
            if len(successors) > 1:
                alternative = successors[1][0]
            else:
                alternative = float('inf')

            visited.add(best_state)

            result, best_f = recursive_rbfs(
                best_state,
                best_g,
                min(f_limit, alternative),
                path + [best_state],
                visited
            )

            if result is not None:
                return result, best_f
🔥 Run It
solution, _ = rbfs(initial, goal)

for step in solution:
    print(step)